In [ ]:
# !pip install yfinance
# !pip install TA-Lib 
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy

In [ ]:
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt
from tqdm import tqdm


In [ ]:
# LOAD STOCK DATA FROM CSV FILE

# Configuration - Change these variables as needed
START_DATE = '2017-01-01'
END_DATE = '2024-06-21'

# Load data from CSV
DATA_PATH = r"data/QQQ.csv"
stock_data = pd.read_csv(DATA_PATH, index_col=0, parse_dates=True)

# Filter by date range
stock_data = stock_data.loc[START_DATE:END_DATE]

if not stock_data.empty:
    print(f"Successfully loaded {len(stock_data)} records")
    print(f"Data range: {stock_data.index.min().date()} to {stock_data.index.max().date()}")
    print("\nFirst 5 rows:")
    print(stock_data.head())
else:
    print(f"Failed to load data from {DATA_PATH}")

# Display the data
stock_data

In [ ]:
# PREPARE PRICE SERIES

warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="invalid value encountered in scalar divide", category=RuntimeWarning)

def select_close_series(df):
    """Select close price column, handling both 'Close' and 'close' column names"""
    if isinstance(df.columns, pd.MultiIndex):
        # Handle MultiIndex columns
        cols = [c for c in df.columns if 'Close' in str(c) or 'close' in str(c)]
        if not cols:
            raise KeyError("Close/close column not found")
        s = df[cols[0]]
    else:
        # Handle regular columns - try 'Close' first, then 'close'
        if 'Close' in df.columns:
            s = df['Close']
        elif 'close' in df.columns:
            s = df['close']
        else:
            raise KeyError("Close/close column not found")
    return s.astype(float).squeeze()

close = select_close_series(stock_data)
close.name = 'price'

# Simple train/validation split
TRAIN_RATIO = 0.60 
split_idx = int(len(close) * TRAIN_RATIO)
train_close = close.iloc[:split_idx].copy()
val_close = close.iloc[split_idx:].copy()

print(f"Data ready: train={train_close.index[0].date()} → {train_close.index[-1].date()} | "
      f"val={val_close.index[0].date()} → {val_close.index[-1].date()}")

MACD CROSSOVER GRID SEARCH - TRAINING SET
----------------------------------------------

This section performs a comprehensive grid search optimization for the **MACD Crossover Strategy** using only the **training data**.

The goal is to find the optimal MACD parameters that maximize the Sharpe ratio on unseen data.

**Strategy Logic**: Buy when MACD line crosses above the signal line. Sell when MACD line crosses below the signal line.

---

In [ ]:
# Define Parameter Ranges for MACD Strategy

# MACD parameters
fast_periods = list(range(8, 20, 2))      # Fast EMA (typically 12)
slow_periods = list(range(20, 35, 3))     # Slow EMA (typically 26)
signal_periods = list(range(7, 13, 1))    # Signal line (typically 9)

print("MACD Fast Periods:")
for i, period in enumerate(fast_periods, 1):
    print(f"  {i}. {period} periods")

print("MACD Slow Periods:")
for i, period in enumerate(slow_periods, 1):
    print(f"  {i}. {period} periods")

print("Signal Line Periods:")
for i, period in enumerate(signal_periods, 1):
    print(f"  {i}. {period} periods")

# Generate all valid combinations (fast < slow)
macd_combinations = []
for fast in fast_periods:
    for slow in slow_periods:
        for signal in signal_periods:
            if fast < slow:
                macd_combinations.append((fast, slow, signal))

print(f"\nGenerated {len(macd_combinations)} valid MACD combinations")
print("\n📋 First 10 combinations preview:")
for i, (fast, slow, signal) in enumerate(macd_combinations[:10], 1):
    print(f"  {i:2d}. Fast: {fast:2d} | Slow: {slow:2d} | Signal: {signal:2d}")
if len(macd_combinations) > 10:
    print(f"   ... and {len(macd_combinations) - 10} more combinations")

print("Ready to test all combinations on training data!")

In [ ]:
# Initialize Results Collection System

grid_search_results = []

print("Strategy Results Collection System Initialized")
print(f"   - Will test {len(macd_combinations)} parameter combinations")
print("   - Results will be stored in 'grid_search_results' list")

metrics_to_collect = [
    # Strategy Parameters
    "fast_period", "slow_period", "signal_period",
    # Return Metrics
    "total_return", "annualized_return", "total_profit",
    # Risk-Adjusted Return Metrics
    "sharpe_ratio", "sortino_ratio", "calmar_ratio",
    "omega_ratio", "tail_ratio",
    # Risk Metrics
    "max_drawdown", "volatility",
    # Trade Performance Metrics
    "win_rate", "total_trades", "expectancy", "profit_factor"
]

print("\nMetrics to collect for each combination:")
for i, metric in enumerate(metrics_to_collect, 1):
    print(f"  {i}. {metric.replace('_', ' ').title()}")

print("Ready to start grid search!")

In [ ]:
# Grid Search Visualization Parameters

FREQ = '1D'  # Daily frequency

print("Visualization Settings:")
print(f"  - Frequency: {FREQ}")
print(f"  - Total combinations to test: {len(macd_combinations)}")
print(f"  - Training data points: {len(train_close)}")
print("\nReady to visualize results!")

In [ ]:
# MACD GRID SEARCH EXECUTION

price_np = train_close.to_numpy(dtype=float)
idx = train_close.index

def series_to_np_bool(s: pd.Series) -> np.ndarray:
    return s.reindex(idx).where(s.reindex(idx).notna(), False).astype(bool).to_numpy(dtype=bool)

grid_search_results = []

print(f"Starting MACD grid search: {len(macd_combinations)} combinations\n")

for fast, slow, signal in tqdm(macd_combinations, desc="Testing MACD combinations"):
    try:
        # Calculate MACD
        macd_line, signal_line, hist = talib.MACD(
            train_close.values, 
            fastperiod=fast, 
            slowperiod=slow, 
            signalperiod=signal
        )
        
        macd_series = pd.Series(macd_line, index=train_close.index)
        signal_series = pd.Series(signal_line, index=train_close.index)
        
        # Generate signals (raw)
        entries_raw = macd_series > signal_series
        entries_raw = entries_raw & (entries_raw.shift(1) == False)  # Crossover detection
        exits_raw = macd_series < signal_series
        exits_raw = exits_raw & (exits_raw.shift(1) == False)
        
        # Fix lookahead bias: shift signals by 1 bar
        entries = entries_raw.shift(1).fillna(False)
        exits = exits_raw.shift(1).fillna(False)
        
        entries_np = series_to_np_bool(entries)
        exits_np = series_to_np_bool(exits)
        
        # Backtest
        pf = vbt.Portfolio.from_signals(
            close=price_np,
            entries=entries_np,
            exits=exits_np,
            init_cash=100_000,
            fees=0.0005,
            slippage=0.0005,
            freq=FREQ
        )
        
        # Calculate metrics
        total_return = float(pf.total_return())
        sharpe = float(pf.sharpe_ratio(freq=FREQ))
        
        if np.isnan(sharpe) or np.isinf(sharpe):
            continue
        
        # Store results
        grid_search_results.append({
            'fast_period': fast,
            'slow_period': slow,
            'signal_period': signal,
            'total_return': total_return,
            'sharpe_ratio': sharpe,
            'sortino_ratio': float(pf.sortino_ratio(freq=FREQ)),
            'calmar_ratio': float(pf.calmar_ratio(freq=FREQ)),
            'max_drawdown': float(pf.max_drawdown()),
            'volatility': float(pf.annualized_volatility(freq=FREQ)),
            'total_trades': len(pf.trades),
        })
        
    except Exception as e:
        continue

results_df = pd.DataFrame(grid_search_results)
print(f"\n✓ Grid search complete: {len(results_df)} valid results")

In [ ]:
# DISPLAY TOP RESULTS

if not results_df.empty:
    top_10 = results_df.nlargest(10, 'sharpe_ratio')
    print("Top 10 Strategies by Sharpe Ratio:\n")
    print(top_10[['fast_period', 'slow_period', 'signal_period', 'sharpe_ratio', 'total_return', 'max_drawdown']].to_string(index=False))
    
    best = top_10.iloc[0]
    print(f"\n--- Best Strategy ---")
    print(f"MACD({best['fast_period']:.0f}, {best['slow_period']:.0f}, {best['signal_period']:.0f})")
    print(f"Sharpe: {best['sharpe_ratio']:.3f} | Return: {best['total_return']:.2%} | DD: {best['max_drawdown']:.2%}")
else:
    print("No valid results found")

In [ ]:
# VALIDATE ON TEST SET

if not results_df.empty:
    best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    fast, slow, signal = int(best['fast_period']), int(best['slow_period']), int(best['signal_period'])
    
    # Test on validation set
    macd_line, signal_line, hist = talib.MACD(val_close.values, fastperiod=fast, slowperiod=slow, signalperiod=signal)
    macd_series = pd.Series(macd_line, index=val_close.index)
    signal_series = pd.Series(signal_line, index=val_close.index)
    
    entries_raw = (macd_series > signal_series) & (macd_series.shift(1) <= signal_series.shift(1))
    exits_raw = (macd_series < signal_series) & (macd_series.shift(1) >= signal_series.shift(1))
    entries = entries_raw.shift(1).fillna(False)
    exits = exits_raw.shift(1).fillna(False)
    
    pf_val = vbt.Portfolio.from_signals(
        close=val_close.to_numpy(dtype=float),
        entries=entries.to_numpy(dtype=bool),
        exits=exits.to_numpy(dtype=bool),
        init_cash=100_000,
        fees=0.0005,
        slippage=0.0005,
        freq=FREQ
    )
    
    print(f"Validation Results - MACD({fast}, {slow}, {signal})")
    print(f"Return: {pf_val.total_return():.2%} | Sharpe: {pf_val.sharpe_ratio(freq=FREQ):.3f}")
    print(f"Max DD: {pf_val.max_drawdown():.2%} | Trades: {len(pf_val.trades)}")

In [ ]:
# TRAIN VS VALIDATION COMPARISON

if not results_df.empty:
    print("Performance Summary:")
    print(f"Train Sharpe: {best['sharpe_ratio']:.3f} | Val Sharpe: {pf_val.sharpe_ratio(freq=FREQ):.3f}")
    print(f"Train Return: {best['total_return']:.2%} | Val Return: {pf_val.total_return():.2%}")
    
    if pf_val.sharpe_ratio(freq=FREQ) > 0.5:
        print("\n✓ Strategy shows positive validation performance")
    else:
        print("\n⚠ Strategy underperforms on validation set")

In [ ]:
# FULL SAMPLE BACKTEST

if not results_df.empty:
    macd_line, signal_line, hist = talib.MACD(close.values, fastperiod=fast, slowperiod=slow, signalperiod=signal)
    macd_series = pd.Series(macd_line, index=close.index)
    signal_series = pd.Series(signal_line, index=close.index)
    
    entries_raw = (macd_series > signal_series) & (macd_series.shift(1) <= signal_series.shift(1))
    exits_raw = (macd_series < signal_series) & (macd_series.shift(1) >= signal_series.shift(1))
    entries = entries_raw.shift(1).fillna(False)
    exits = exits_raw.shift(1).fillna(False)
    
    pf_full = vbt.Portfolio.from_signals(
        close=close.to_numpy(dtype=float),
        entries=entries.to_numpy(dtype=bool),
        exits=exits.to_numpy(dtype=bool),
        init_cash=100_000,
        fees=0.0005,
        slippage=0.0005,
        freq=FREQ
    )
    
    print(f"Full Sample Results - MACD({fast}, {slow}, {signal})")
    print(f"Return: {pf_full.total_return():.2%} | Sharpe: {pf_full.sharpe_ratio(freq=FREQ):.3f}")
    print(f"Max DD: {pf_full.max_drawdown():.2%} | Trades: {len(pf_full.trades)}")
    
    # Plot equity curve
    pf_full.plot().show()

In [ ]:
# TRADE-BY-TRADE ANALYSIS

if not results_df.empty and len(pf_full.trades) > 0:
    trades = pf_full.trades
    trade_returns = trades.returns.values if hasattr(trades.returns, 'values') else np.asarray(trades.returns)
    trade_returns = np.asarray(trade_returns).ravel()
    
    winning_trades = trade_returns[trade_returns > 0]
    losing_trades = trade_returns[trade_returns < 0]
    
    total_trades = len(trade_returns)
    win_count = len(winning_trades)
    loss_count = len(losing_trades)
    win_rate = (win_count / total_trades * 100) if total_trades > 0 else 0
    
    print(f"Trade Statistics:")
    print(f"Total: {total_trades} | Win Rate: {win_rate:.1f}% ({win_count}W/{loss_count}L)")
    print(f"Avg Win: {winning_trades.mean()*100:.2f}% | Avg Loss: {losing_trades.mean()*100:.2f}%")
    
    # Plot trade returns
    fig, ax = plt.subplots(figsize=(14, 6))
    colors = ['green' if r > 0 else 'red' for r in trade_returns]
    ax.bar(range(len(trade_returns)), trade_returns * 100, color=colors, alpha=0.7)
    ax.axhline(0, color='black', linewidth=1)
    ax.set_title('Per-Trade Returns (%)')
    ax.set_xlabel('Trade #')
    ax.set_ylabel('Return (%)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# PARAMETER SENSITIVITY ANALYSIS

if not results_df.empty:
    best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    fast, slow, signal = int(best['fast_period']), int(best['slow_period']), int(best['signal_period'])
    
    print(f"Sensitivity Analysis - MACD({fast}, {slow}, {signal})")
    print("="*60)
    
    # Test variations
    fast_range = list(range(max(2, fast-5), fast+6))
    slow_range = list(range(max(3, slow-5), slow+6))
    signal_range = list(range(max(2, signal-3), signal+4))
    
    sensitivity_results = []
    for f in fast_range:
        for sl in slow_range:
            for si in signal_range:
                if f < sl:
                    try:
                        macd_line, signal_line, _ = talib.MACD(train_close.values, fastperiod=f, slowperiod=sl, signalperiod=si)
                        macd_s = pd.Series(macd_line, index=train_close.index)
                        signal_s = pd.Series(signal_line, index=train_close.index)
                        
                        e = ((macd_s > signal_s) & (macd_s.shift(1) <= signal_s.shift(1))).shift(1).fillna(False)
                        x = ((macd_s < signal_s) & (macd_s.shift(1) >= signal_s.shift(1))).shift(1).fillna(False)
                        
                        pf_sens = vbt.Portfolio.from_signals(
                            close=train_close.to_numpy(dtype=float),
                            entries=e.to_numpy(dtype=bool),
                            exits=x.to_numpy(dtype=bool),
                            init_cash=100_000,
                            fees=0.0005,
                            slippage=0.0005,
                            freq=FREQ
                        )
                        
                        sharpe_sens = float(pf_sens.sharpe_ratio(freq=FREQ))
                        if not np.isnan(sharpe_sens) and not np.isinf(sharpe_sens):
                            sensitivity_results.append({
                                'fast': f, 'slow': sl, 'signal': si,
                                'sharpe': sharpe_sens,
                                'return': float(pf_sens.total_return())
                            })
                    except:
                        pass
    
    if sensitivity_results:
        sens_df = pd.DataFrame(sensitivity_results)
        print(f"\nTested {len(sens_df)} parameter variations")
        print(f"Sharpe range: {sens_df['sharpe'].min():.3f} to {sens_df['sharpe'].max():.3f}")
        print(f"Best params remain stable: {sens_df['sharpe'].idxmax() == 0}")